> **Open in VS Code:** From your cloned `s26-06643` repository, run in the terminal:
> ```bash
> code sse/optional/release-lifecycle/release-lifecycle.ipynb
> ```
> [View source on GitHub](https://github.com/jkitchin/s26-06643/blob/main/sse/optional/release-lifecycle/release-lifecycle.ipynb)

# Release Lifecycle: From Commits to PyPI

You have learned to write packages, test them, lint them, and push them to GitHub. But how does working code become a *release* that others can `pip install`? This module covers the complete release lifecycle: deciding version numbers, generating changelogs, publishing to PyPI, and automating the entire process so that tagging a commit is all it takes to ship a new version.

## Section 1: Semantic Versioning in Practice

You have seen semantic versioning mentioned as a concept: `MAJOR.MINOR.PATCH`. But knowing the format and knowing how to *use* it are different things. This section teaches you how to make versioning decisions.

### The Rules

Given a version number `MAJOR.MINOR.PATCH`:

- **PATCH** (e.g., 1.2.3 → 1.2.4): Bug fixes, typo corrections, documentation fixes. No behavior change for correct usage.
- **MINOR** (e.g., 1.2.4 → 1.3.0): New features, new functions, new optional parameters. Existing code still works.
- **MAJOR** (e.g., 1.3.0 → 2.0.0): Breaking changes. Existing code may need modification.

### What Counts as a Breaking Change?

A change is breaking if code that worked with the previous version stops working (raises errors, returns different results, or changes behavior) without the user changing anything. Examples:

| Change | Breaking? | Version Bump |
|--------|-----------|-------------|
| Rename a public function | Yes | MAJOR |
| Remove a function parameter | Yes | MAJOR |
| Change default return type | Yes | MAJOR |
| Add a new optional parameter | No | MINOR |
| Add a new function | No | MINOR |
| Fix a bug that returned wrong results | No* | PATCH |
| Improve performance | No | PATCH |
| Add a new module | No | MINOR |

\* If someone depended on the buggy behavior, this is technically breaking. Use judgment.

### Pre-release Versions

Before your first stable release, you can use `0.x.y` versions:

- `0.1.0` — initial development, anything may change
- `0.2.0` — still unstable, breaking changes are expected
- `1.0.0` — your first stable release, a commitment to backward compatibility

For release candidates and pre-releases:

```
1.0.0a1    # alpha 1 — early testing
1.0.0b1    # beta 1 — feature complete, may have bugs
1.0.0rc1   # release candidate 1 — should be final unless bugs found
1.0.0      # stable release
```

### Where the Version Lives

Your version number should be defined in **one place**. The standard location is `pyproject.toml`:

```toml
[project]
name = "mypackage"
version = "0.3.1"
```

You can also use dynamic versioning (reading from a `__version__` attribute or git tags), but a static version in `pyproject.toml` is simplest to start with.

### Exercise 1: Version Bump Decisions (5 min)

Your package `scholartools` is at version `1.2.0`. For each change below, decide the new version number and justify your choice:

1. You fix a bug where `get_author()` crashed on authors with no institution.
2. You add a new function `get_institution(name)` that returns institution metadata.
3. You rename `search_works()` to `search_publications()` because it is more descriptive.
4. You add an optional `sort_by` parameter to `search_works()`.
5. You change `get_citation_counts()` to return a DataFrame instead of a list of dicts.

In [ ]:
# Write your version decisions here:
# 1. Bug fix: 1.2.0 → ?
# 2. New function: ? → ?
# 3. Renamed function: ? → ?
# 4. New optional param: ? → ?
# 5. Changed return type: ? → ?

## Section 2: Git Tags and GitHub Releases

A git tag marks a specific commit as a release point. Tags are the bridge between your git history and your version numbers.

### Creating Tags

```bash
# Create an annotated tag (preferred — includes metadata)
git tag -a v1.0.0 -m "Release version 1.0.0"

# List tags
git tag

# Push tags to GitHub
git push origin v1.0.0

# Push all tags at once
git push --tags
```

**Convention:** Prefix version tags with `v` (e.g., `v1.0.0`). This distinguishes version tags from other tags and is expected by most automation tools.

### The Tag-Version Contract

Your workflow should be:

1. Update the version in `pyproject.toml`
2. Commit: `git commit -m "chore: bump version to 1.1.0"`
3. Tag: `git tag -a v1.1.0 -m "Release 1.1.0"`
4. Push: `git push origin main --tags`

The tag and the version in `pyproject.toml` **must match**. If they diverge, you will ship confusing releases.

### GitHub Releases

A GitHub Release is a UI wrapper around a git tag. It adds:
- A title and description (release notes)
- Downloadable source archives (.tar.gz, .zip)
- The ability to attach binary assets
- Auto-generated release notes from PR titles and commit messages

To create a GitHub Release:

```bash
# Using the GitHub CLI
gh release create v1.1.0 --title "v1.1.0" --generate-notes

# Or from the web UI:
# 1. Go to your repo → Releases → "Draft a new release"
# 2. Choose the tag
# 3. Click "Generate release notes" (auto-generates from PRs)
# 4. Click "Publish release"
```

The `--generate-notes` flag uses your merged PR titles and commit messages to create release notes automatically. This is where conventional commits pay off — `feat:`, `fix:`, and `docs:` prefixes make auto-generated notes readable.

### Exercise 2: Create a Tag and GitHub Release (10 min)

Using any repository you have on GitHub (your course project or a practice repo):

1. Check your current version in `pyproject.toml`
2. Create an annotated git tag matching that version: `git tag -a v0.1.0 -m "Initial release"`
3. Push the tag: `git push origin v0.1.0`
4. Create a GitHub Release using the CLI: `gh release create v0.1.0 --title "v0.1.0" --generate-notes`
5. View your release on GitHub and examine the auto-generated notes

If you used conventional commit messages, notice how the auto-generated notes organize changes by type.

## Section 3: Changelogs

A changelog tells your users what changed between versions. It answers the question: *"Should I upgrade, and if so, what do I need to know?"*

### The Keep a Changelog Format

The most widely used format is [Keep a Changelog](https://keepachangelog.com/). A `CHANGELOG.md` file at the root of your repository:

```markdown
# Changelog

All notable changes to this project will be documented in this file.

## [1.1.0] - 2026-03-15

### Added
- `get_institution()` function for looking up institution metadata
- Optional `sort_by` parameter to `search_works()`

### Fixed
- `get_author()` no longer crashes when author has no institution

## [1.0.0] - 2026-03-01

### Added
- Initial release with `search_works()`, `get_author()`, and `get_citation_counts()`
```

The categories are: **Added**, **Changed**, **Deprecated**, **Removed**, **Fixed**, **Security**.

### Why Not Just Use Git Log?

Your git log is for developers. Your changelog is for users. The differences:

| Git Log | Changelog |
|---------|----------|
| Every commit | Only notable changes |
| Developer-facing | User-facing |
| Includes refactors, typos, CI fixes | Only what affects users |
| Organized by time | Organized by version |

### Generating Changelogs from Conventional Commits

If you have been writing conventional commit messages (as taught in the git module), you can auto-generate changelogs. The tool [git-cliff](https://git-cliff.org/) does this:

```bash
# Install git-cliff
pip install git-cliff

# Generate a changelog from your commit history
git-cliff --output CHANGELOG.md

# Generate for a specific version range
git-cliff v1.0.0..v1.1.0 --output CHANGELOG.md
```

git-cliff maps conventional commit prefixes to changelog categories:

| Commit Prefix | Changelog Category |
|--------------|-------------------|
| `feat:` | Added |
| `fix:` | Fixed |
| `docs:` | Documentation |
| `refactor:` | Changed |
| `perf:` | Performance |
| `BREAKING CHANGE:` | Breaking |

This is the payoff for writing conventional commits — they are not just for readability, they are structured data that tools can consume.

### Exercise 3: Generate a Changelog (10 min)

Using a repository with conventional commit messages:

1. Install git-cliff: `pip install git-cliff`
2. Run `git-cliff` in your repository and examine the output
3. Save it: `git-cliff --output CHANGELOG.md`
4. Open `CHANGELOG.md` and review how your commits were categorized
5. If some commits were miscategorized or missing, consider how you might improve your commit messages

If your repository does not use conventional commits, try `git-cliff` anyway — you will see how unhelpful the output is, which demonstrates why the convention matters.

## Section 4: Publishing to PyPI

Publishing a package makes it available to the world via `pip install yourpackage`. This section walks through the full process.

### Step 1: Create a PyPI Account

1. Go to [https://pypi.org/account/register/](https://pypi.org/account/register/) and create an account
2. Go to [https://test.pypi.org/account/register/](https://test.pypi.org/account/register/) and create a TestPyPI account (separate registration)

TestPyPI is a separate instance of PyPI for testing. Always publish there first.

### Step 2: Build Your Package

```bash
# Make sure build is installed
pip install build

# Build the package (creates dist/ directory)
python -m build
```

This creates two files in `dist/`:
- `yourpackage-0.1.0.tar.gz` — source distribution
- `yourpackage-0.1.0-py3-none-any.whl` — wheel (pre-built)

### Step 3: Upload to TestPyPI

```bash
# Install twine (the upload tool)
pip install twine

# Upload to TestPyPI
twine upload --repository testpypi dist/*
```

You will be prompted for your TestPyPI username and password (or API token).

### Step 4: Verify the Upload

```bash
# Install from TestPyPI to verify it works
pip install --index-url https://test.pypi.org/simple/ yourpackage

# Test it
python -c "import yourpackage; print(yourpackage.__version__)"
```

If the install works and the import succeeds, you are ready for the real thing.

### Step 5: Upload to PyPI

```bash
# Upload to real PyPI
twine upload dist/*
```

Your package is now available to the world via `pip install yourpackage`.

### Common Pitfalls

- **Version already exists**: PyPI does not allow re-uploading the same version. If you need to fix something, bump the version number.
- **Name already taken**: Check [pypi.org](https://pypi.org) before choosing a package name.
- **Missing metadata**: PyPI requires `name`, `version`, and a few other fields. Your `pyproject.toml` should have them.

### Exercise 4: Publish to TestPyPI (15 min)

Using your course project or a practice package:

1. Ensure your `pyproject.toml` has all required fields (`name`, `version`, `description`, `readme`, `license`, `requires-python`)
2. Build: `python -m build`
3. Create a TestPyPI account if you do not have one
4. Upload: `twine upload --repository testpypi dist/*`
5. Install from TestPyPI in a fresh virtual environment:
   ```bash
   python -m venv /tmp/test-install
   source /tmp/test-install/bin/activate
   pip install --index-url https://test.pypi.org/simple/ yourpackage
   python -c "import yourpackage"
   ```
6. Verify it works, then deactivate and clean up

**Note:** If your package has dependencies from real PyPI (like `requests`), you may need:
```bash
pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ yourpackage
```

## Section 5: Automated Publishing with GitHub Actions

Manually running `twine upload` works, but it is error-prone and depends on you having credentials configured locally. The modern approach is to automate publishing through CI/CD, triggered by creating a GitHub Release.

### Trusted Publishers (OIDC)

PyPI supports [Trusted Publishers](https://docs.pypi.org/trusted-publishers/), which use OpenID Connect (OIDC) to authenticate GitHub Actions without API tokens. This is the recommended approach because:

- No secrets to manage or rotate
- No tokens that could leak
- PyPI verifies the request came from your specific GitHub repo and workflow

### Setting Up Trusted Publishing

1. Go to your PyPI project → "Publishing" → "Add a new publisher"
2. Fill in:
   - Owner: your GitHub username
   - Repository: your repo name
   - Workflow name: `publish.yml`
   - Environment name: `pypi` (optional but recommended)

### The Workflow

Create `.github/workflows/publish.yml`:

```yaml
name: Publish to PyPI

on:
  release:
    types: [published]

jobs:
  publish:
    runs-on: ubuntu-latest
    environment: pypi
    permissions:
      id-token: write  # Required for trusted publishing
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install build tools
        run: pip install build

      - name: Build package
        run: python -m build

      - name: Publish to PyPI
        uses: pypa/gh-action-pypi-publish@release/v1
```

Notice: no API tokens, no secrets. The `pypa/gh-action-pypi-publish` action handles OIDC authentication automatically.

### The Full Release Flow

With this workflow in place, releasing a new version is:

1. Update version in `pyproject.toml`
2. Update `CHANGELOG.md` (or generate with git-cliff)
3. Commit and push
4. Create a git tag: `git tag -a v1.1.0 -m "Release 1.1.0"`
5. Push the tag: `git push origin v1.1.0`
6. Create a GitHub Release: `gh release create v1.1.0 --generate-notes`
7. CI automatically builds and publishes to PyPI

Steps 4-7 can be reduced to a single command:

```bash
git tag -a v1.1.0 -m "Release 1.1.0" && git push origin v1.1.0 && gh release create v1.1.0 --generate-notes
```

## Section 6: Dependency Specification Strategies

When your package declares dependencies, the version constraints you choose determine whether your package plays well with others.

### Libraries vs. Applications

There are two fundamentally different approaches to dependency specification:

**Libraries** (installed by others as a dependency):
- Use **loose** version constraints
- Goal: maximize compatibility with other packages

```toml
# Good for a library
dependencies = [
    "numpy>=1.24",
    "requests>=2.28",
]
```

**Applications** (deployed or run directly):
- Use **locked** versions via a lock file
- Goal: reproducible deployments

```
# requirements.txt (locked) or uv.lock
numpy==2.1.3
requests==2.31.0
```

### Version Constraint Operators

| Operator | Meaning | Example | Matches |
|----------|---------|---------|--------|
| `>=` | At least | `numpy>=1.24` | 1.24, 1.25, 2.0, ... |
| `==` | Exactly | `numpy==2.0.0` | Only 2.0.0 |
| `~=` | Compatible | `numpy~=1.24` | >=1.24, <2.0 |
| `<` | Less than | `numpy<2.0` | 1.x only |
| `>=,<` | Range | `numpy>=1.24,<2.0` | 1.24 through 1.x |

### Upper Bounds: Handle with Care

Adding `<2.0` to your dependency means your package *cannot be installed* alongside anything that requires numpy 2.0+. This causes dependency conflicts for your users.

**When to use upper bounds:**
- You have *tested* and *know* your code breaks with the newer version
- The dependency has a history of breaking changes

**When NOT to use upper bounds:**
- Preemptively, "just in case"
- Because it "feels safer"

The general rule: **libraries should use lower bounds only** (`>=1.24`), unless you have a specific reason for an upper bound.

### The `requires-python` Field

```toml
[project]
requires-python = ">=3.10"
```

This tells pip which Python versions your package supports. Set it to the oldest version you test against in CI. If your CI matrix tests 3.10, 3.11, and 3.12, use `>=3.10`.

### Exercise 5: Dependency Audit (10 min)

Examine the following `pyproject.toml` dependencies and identify problems:

```toml
[project]
name = "chemtools"
version = "0.2.0"
requires-python = ">=3.8"
dependencies = [
    "numpy==1.24.3",
    "scipy>=1.10,<1.12",
    "matplotlib",
    "pandas>=2.0,<2.1",
    "requests",
]
```

For each dependency, answer:
1. Is this constraint appropriate for a library? Why or why not?
2. What problems could this cause for users?
3. What would you change?

In [ ]:
# Write your analysis here

## Section 7: Package Metadata and Discoverability

When someone finds your package on PyPI, your metadata determines whether they trust it enough to install. A minimal `pyproject.toml` works, but a well-populated one signals professionalism.

### Essential Metadata

```toml
[project]
name = "scholartools"
version = "1.0.0"
description = "Python tools for querying academic publication data via OpenAlex"
readme = "README.md"
license = {text = "MIT"}
requires-python = ">=3.10"
authors = [
    {name = "Your Name", email = "you@example.com"},
]
```

### Classifiers

Classifiers are standardized tags that help users find your package on PyPI:

```toml
classifiers = [
    "Development Status :: 3 - Alpha",
    "Intended Audience :: Science/Research",
    "License :: OSI Approved :: MIT License",
    "Programming Language :: Python :: 3",
    "Programming Language :: Python :: 3.10",
    "Programming Language :: Python :: 3.11",
    "Programming Language :: Python :: 3.12",
    "Topic :: Scientific/Engineering",
]
```

The full list is at [pypi.org/classifiers](https://pypi.org/classifiers/). The most important one is `Development Status` — it tells users how mature your package is.

### Project URLs

These appear as links on your PyPI page:

```toml
[project.urls]
Homepage = "https://github.com/you/scholartools"
Documentation = "https://scholartools.readthedocs.io"
Repository = "https://github.com/you/scholartools"
Changelog = "https://github.com/you/scholartools/blob/main/CHANGELOG.md"
"Issue Tracker" = "https://github.com/you/scholartools/issues"
```

### Keywords

```toml
keywords = ["openalex", "scholarly", "citations", "academic", "publications"]
```

Keywords help with PyPI search. Choose terms your users would search for.

## Section 8: Deprecation — Removing Things Gracefully

Once people depend on your package, you cannot just remove or rename things without warning. Python provides `warnings.warn` for this:

```python
import warnings

def search_works(query, limit=10):
    """Search for academic papers."""
    # ... implementation ...


def search_papers(query, limit=10):
    """Deprecated: Use search_works() instead."""
    warnings.warn(
        "search_papers() is deprecated, use search_works() instead. "
        "It will be removed in version 2.0.0.",
        DeprecationWarning,
        stacklevel=2,
    )
    return search_works(query, limit)
```

### The Deprecation Timeline

1. **Version 1.1.0**: Add `search_works()`, deprecate `search_papers()` with a warning
2. **Version 1.2.0, 1.3.0, ...**: Warning continues, old function still works
3. **Version 2.0.0**: Remove `search_papers()` (this is a breaking change → MAJOR bump)

Users see the warning, have time to update their code, and are not surprised when the removal happens.

### Testing Deprecation Warnings

```python
import pytest

def test_search_papers_deprecated():
    with pytest.warns(DeprecationWarning, match="use search_works"):
        search_papers("test")
```

## Tips and Tricks

- **Start at `0.1.0`**, not `1.0.0`. The `0.x` range signals that your API is not yet stable. Bump to `1.0.0` when you are confident in your public API.
- **Tag every release in git.** A version in `pyproject.toml` without a corresponding git tag is invisible to tools and collaborators.
- **Use `gh release create` with `--generate-notes`** to get free release notes from your PR titles and commit messages.
- **Set up trusted publishing on PyPI** before your first release. It takes 5 minutes and eliminates API token management forever.
- **Write your changelog as you go**, not at release time. Add a note to `CHANGELOG.md` with every PR, or use git-cliff to generate it.
- **Never use `==` in library dependencies** unless you have a very specific reason. Exact pins cause dependency conflicts for your users.
- **Test installation from TestPyPI before publishing to real PyPI.** You cannot delete or re-upload a version on PyPI.
- **Ask your AI agent to draft release notes** from your recent git log — it is good at summarizing changes into user-facing descriptions.
- **Add classifiers and project URLs to `pyproject.toml`** before your first PyPI upload. They cost nothing and significantly improve discoverability.

## Foundational Concepts to Commit to Memory

- **Semantic versioning (`MAJOR.MINOR.PATCH`) communicates the nature of changes** — PATCH for fixes, MINOR for additions, MAJOR for breaking changes.
- **Git tags mark release points** and should always match the version in `pyproject.toml`. Use annotated tags with the `v` prefix (e.g., `v1.0.0`).
- **GitHub Releases wrap git tags** with release notes and downloadable assets. Use `--generate-notes` to auto-generate notes from conventional commits.
- **Changelogs are for users, git logs are for developers.** Use tools like git-cliff to generate changelogs from conventional commit messages.
- **Always publish to TestPyPI first** to verify your package installs correctly before publishing to real PyPI.
- **Trusted publishers (OIDC) replace API tokens** for automated PyPI publishing from GitHub Actions — no secrets to manage.
- **Libraries use loose dependency constraints** (`>=1.24`); **applications use locked versions** (lock files). Upper bounds on library dependencies cause conflicts.
- **Deprecation warnings give users time to migrate** before you remove functionality. Warn for at least one minor version before removing in a major version.

## Knowing vs. Doing: Reflection

Version numbers seem trivial until you ship a breaking change as a patch and break someone's CI pipeline. Changelogs seem like busywork until you are the user trying to decide whether to upgrade a dependency. Publishing seems like the last step until you realize you uploaded a broken version to PyPI and cannot re-upload the same number.

The release lifecycle is where software engineering meets communication. Every version number is a promise to your users about what changed. Every changelog entry is a signal about what you considered important. Every dependency constraint is a statement about how your package fits into the broader ecosystem.

Automating these processes is not about saving the 30 seconds it takes to run `twine upload`. It is about removing the human decisions that should not be made under time pressure at 5 PM on a Friday. When your release workflow is: tag, push, done — you are more likely to release often, and frequent small releases are always less risky than infrequent large ones.

The conventional commits you write, the version numbers you choose, the changelog entries you maintain, and the dependency constraints you set — these are all forms of communication with your future self, your collaborators, and your users. Get them right, and people will trust your package. Get them wrong, and they will find an alternative.

## Additional Resources

- [Semantic Versioning 2.0.0](https://semver.org/) — the full specification for semantic versioning
- [Keep a Changelog](https://keepachangelog.com/) — the standard format for maintaining changelogs
- [git-cliff Documentation](https://git-cliff.org/) — auto-generate changelogs from conventional commits
- [PyPI Trusted Publishers](https://docs.pypi.org/trusted-publishers/) — set up token-free publishing from GitHub Actions
- [pypa/gh-action-pypi-publish](https://github.com/pypa/gh-action-pypi-publish) — the official GitHub Action for publishing to PyPI
- [Python Packaging User Guide](https://packaging.python.org/) — the authoritative guide to packaging Python projects
- [GitHub CLI (`gh`) Manual](https://cli.github.com/manual/) — documentation for creating releases and tags from the command line